In [2]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../Data/Raw")

df_german = pd.read_csv(DATA_DIR / "German Credit Risk Dataset" / "german_credit_uci_original.csv")
df_home = pd.read_csv(DATA_DIR / "Home Credit Default Risk Dataset" / "application_train_cleaned.csv")
df_lending = pd.read_csv(DATA_DIR / "Lending Club Loan Dataset" / "loans_full_schema_cleaned.csv")

for name, df in [("German Credit", df_german), ("Home Credit", df_home), ("LendingClub", df_lending)]:
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

German Credit: 1000 rows, 21 columns
Home Credit: 307511 rows, 74 columns
LendingClub: 10000 rows, 56 columns


In [3]:
# Reload German Credit fresh - undoes whatever the accidental overwrite did
df_german = pd.read_csv(DATA_DIR / "German Credit Risk Dataset" / "german_credit_uci_original.csv")
print(df_german.shape)  # should show (1000, 21)

(1000, 21)


In [4]:
# Recode target to 0/1
df_german["target"] = df_german["target"].map({1: 0, 2: 1})
print(df_german["target"].value_counts())

target
0    700
1    300
Name: count, dtype: int64


In [5]:
# Cap outliers, lower bound floored at 0
continuous_cols = ["duration_months", "credit_amount", "age"]

for col in continuous_cols:
    Q1 = df_german[col].quantile(0.25)
    Q3 = df_german[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = max(0, Q1 - 1.5 * IQR)
    upper = Q3 + 1.5 * IQR
    n_capped = ((df_german[col] < lower) | (df_german[col] > upper)).sum()
    df_german[col] = df_german[col].clip(lower, upper)
    print(f"{col}: capped {n_capped} outlier values to range [{lower:.1f}, {upper:.1f}]")

duration_months: capped 70 outlier values to range [0.0, 42.0]
credit_amount: capped 72 outlier values to range [0.0, 7882.4]
age: capped 23 outlier values to range [4.5, 64.5]


In [6]:
# Ordinal encoding
savings_order = {
    "unknown/no savings account": 0, "< 100 DM": 1, "100-500 DM": 2,
    "500-1000 DM": 3, ">= 1000 DM": 4
}
employment_order = {
    "unemployed": 0, "< 1 year": 1, "1-4 years": 2, "4-7 years": 3, ">= 7 years": 4
}
job_order = {
    "unemployed/unskilled non-resident": 0, "unskilled resident": 1,
    "skilled employee/official": 2, "management/highly qualified": 3
}

df_german["savings_account"] = df_german["savings_account"].map(savings_order)
df_german["employment_since"] = df_german["employment_since"].map(employment_order)
df_german["job"] = df_german["job"].map(job_order)

print(df_german[["savings_account", "employment_since", "job"]].isnull().sum())

savings_account     0
employment_since    0
job                 0
dtype: int64


In [7]:
# One-hot encode nominal columns
nominal_cols = [
    "status_checking_account", "credit_history", "purpose", "personal_status_sex",
    "other_debtors_guarantors", "property", "other_installment_plans",
    "housing", "telephone", "foreign_worker"
]

df_german = pd.get_dummies(df_german, columns=nominal_cols, drop_first=True)
print(df_german.shape)

(1000, 41)


In [8]:
# Save the fully encoded version
df_german.to_csv(DATA_DIR / "German Credit Risk Dataset" / "german_credit_encoded.csv", index=False)
print("Saved:", df_german.shape)

Saved: (1000, 41)


In [9]:
# Cell 2: check target balance
print(df_home["TARGET"].value_counts())

TARGET
0    282686
1     24825
Name: count, dtype: int64


In [10]:
# Cell 3: drop the 4 rows with bad gender value
df_home = df_home[df_home["CODE_GENDER"] != "XNA"]
print(df_home.shape)

(307507, 74)


In [11]:
# Cell 4: encode simple yes/no columns as 0/1
df_home["NAME_CONTRACT_TYPE"] = df_home["NAME_CONTRACT_TYPE"].map({"Cash loans": 0, "Revolving loans": 1})
df_home["FLAG_OWN_CAR"] = df_home["FLAG_OWN_CAR"].map({"N": 0, "Y": 1})
df_home["FLAG_OWN_REALTY"] = df_home["FLAG_OWN_REALTY"].map({"N": 0, "Y": 1})

In [12]:
# Cell 5: one-hot encode the smaller categorical columns
low_card_cols = [
    "CODE_GENDER", "NAME_TYPE_SUITE", "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE", "WEEKDAY_APPR_PROCESS_START"
]
df_home = pd.get_dummies(df_home, columns=low_card_cols, drop_first=True)
print(df_home.shape)

(307507, 101)


In [13]:
# Cell 6: encode the two big categorical columns by frequency instead
for col in ["ORGANIZATION_TYPE", "OCCUPATION_TYPE"]:
    freq_map = df_home[col].value_counts(normalize=True)
    df_home[col] = df_home[col].map(freq_map)
print(df_home.shape)

(307507, 101)


In [14]:
# Confirm no unexpected nulls were introduced anywhere during encoding
print(df_home.isnull().sum().sum())

# Confirm the binary columns actually got mapped to 0/1, not left as text/NaN
print(df_home[["NAME_CONTRACT_TYPE", "FLAG_OWN_CAR", "FLAG_OWN_REALTY"]].head())

0
   NAME_CONTRACT_TYPE  FLAG_OWN_CAR  FLAG_OWN_REALTY
0                   0             0                1
1                   0             0                0
2                   1             1                1
3                   0             0                1
4                   0             0                1


In [15]:
# Save the encoded Home Credit dataset
df_home.to_csv(DATA_DIR / "Home Credit Default Risk Dataset" / "application_train_encoded.csv", index=False)
print("Saved:", df_home.shape)

Saved: (307507, 101)


In [16]:
# DAYS_EMPLOYED is known to contain a placeholder value (365243) for
# retired/unemployed applicants instead of a real negative day-count.
# Check how many rows are affected before deciding how to handle it.
print(df_home["DAYS_EMPLOYED"].describe())
print((df_home["DAYS_EMPLOYED"] == 365243).sum())

count    307507.000000
mean      63815.929208
std      141276.472519
min      -17912.000000
25%       -2760.000000
50%       -1213.000000
75%        -289.000000
max      365243.000000
Name: DAYS_EMPLOYED, dtype: float64
55374


In [17]:
# DAYS_EMPLOYED contains a placeholder value (365243) for retired/unemployed
# applicants, rather than a real negative day-count. This is a documented
# data quality artifact in this dataset, not genuine missingness.
# Step 1: create a flag column preserving this information as a feature
df_home["DAYS_EMPLOYED_ANOMALY"] = (df_home["DAYS_EMPLOYED"] == 365243).astype(int)

# Step 2: replace the placeholder with 0, representing "not currently employed"
# (consistent with how we handled OWN_CAR_AGE - 0 for "not applicable" rather
# than treating it as a real, extreme value or blindly imputing a median that
# would be distorted by the placeholder rows if left in)
df_home.loc[df_home["DAYS_EMPLOYED"] == 365243, "DAYS_EMPLOYED"] = 0

# Confirm the fix
print(df_home["DAYS_EMPLOYED"].describe())
print(df_home["DAYS_EMPLOYED_ANOMALY"].value_counts())

count    307507.000000
mean      -1954.820342
std        2307.036831
min      -17912.000000
25%       -2760.000000
50%       -1213.000000
75%        -289.000000
max           0.000000
Name: DAYS_EMPLOYED, dtype: float64
DAYS_EMPLOYED_ANOMALY
0    252133
1     55374
Name: count, dtype: int64


In [18]:
# Check summary stats for the main continuous numeric columns to decide
# which need outlier capping
continuous_cols_home = [
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
    "DAYS_BIRTH", "DAYS_EMPLOYED", "DAYS_REGISTRATION", "DAYS_ID_PUBLISH"
]
df_home[continuous_cols_home].describe()

,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH
count,3.075070e+05,3.075070e+05,307507.000000,3.075070e+05,307507.000000,307507.000000,307507.000000,307507.000000
mean,1.687977e+05,5.990286e+05,27108.580714,5.383178e+05,-16037.027271,-1954.820342,-4986.131376,-2994.201670
std,2.371246e+05,4.024926e+05,14493.522125,3.692898e+05,4363.982424,2307.036831,3522.883030,1509.454566
min,2.565000e+04,4.500000e+04,1615.500000,4.050000e+04,-25229.000000,-17912.000000,-24672.000000,-7197.000000
25%,1.125000e+05,2.700000e+05,16524.000000,2.385000e+05,-19682.000000,-2760.000000,-7479.500000,-4299.000000
50%,1.471500e+05,5.135310e+05,24903.000000,4.500000e+05,-15750.000000,-1213.000000,-4504.000000,-3254.000000
75%,2.025000e+05,8.086500e+05,34596.000000,6.795000e+05,-12413.000000,-289.000000,-2010.000000,-1720.000000
max,1.170000e+08,4.050000e+06,258025.500000,4.050000e+06,-7489.000000,0.000000,0.000000,0.000000


In [19]:
# Apply IQR-based outlier capping only to genuinely continuous monetary columns.
# DAYS_BIRTH/DAYS_REGISTRATION/DAYS_ID_PUBLISH are deliberately excluded:
# their negative values are correct (days before application), and their
# ranges are already sensible - capping would incorrectly clip legitimate
# very old/young applicants rather than remove genuine data errors.
monetary_cols = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"]

for col in monetary_cols:
    Q1 = df_home[col].quantile(0.25)
    Q3 = df_home[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = max(0, Q1 - 1.5 * IQR)  # floored at 0 - these cannot be negative
    upper = Q3 + 1.5 * IQR
    n_capped = ((df_home[col] < lower) | (df_home[col] > upper)).sum()
    df_home[col] = df_home[col].clip(lower, upper)
    print(f"{col}: capped {n_capped} outlier values to range [{lower:.1f}, {upper:.1f}]")

AMT_INCOME_TOTAL: capped 14035 outlier values to range [0.0, 337500.0]
AMT_CREDIT: capped 6562 outlier values to range [0.0, 1616625.0]
AMT_ANNUITY: capped 7504 outlier values to range [0.0, 61704.0]
AMT_GOODS_PRICE: capped 14728 outlier values to range [0.0, 1341000.0]


In [20]:
# Save the fully processed Home Credit dataset (encoded + outlier-capped)
df_home.to_csv(DATA_DIR / "Home Credit Default Risk Dataset" / "application_train_encoded.csv", index=False)
print("Saved:", df_home.shape)

Saved: (307507, 102)


In [21]:
# Identify categorical columns and their cardinality
cat_cols_lending = df_lending.select_dtypes(include=["object", "str"]).columns.tolist()

for col in cat_cols_lending:
    print(f"{col}: {df_lending[col].nunique()} unique values")

emp_title: 4742 unique values
state: 50 unique values
homeownership: 3 unique values
verified_income: 3 unique values
verification_income_joint: 4 unique values
loan_purpose: 12 unique values
application_type: 2 unique values
grade: 7 unique values
sub_grade: 32 unique values
issue_month: 3 unique values
loan_status: 6 unique values
initial_listing_status: 2 unique values
disbursement_method: 2 unique values


In [22]:
# Check what the 6 loan_status categories actually are - this determines
# how we construct the binary target (default vs non-default)
print(df_lending["loan_status"].value_counts())

loan_status
Current               9375
Fully Paid             447
In Grace Period         67
Late (31-120 days)      66
Late (16-30 days)       38
Charged Off              7
Name: count, dtype: int64


In [23]:
# Binary target definition following common practice in LendingClub research
# (e.g. multiple independent studies group any sign of repayment trouble as
# "risk", treating only Fully Paid + Current as non-default). This differs
# from the stricter "only fully resolved loans" approach used in some academic
# papers (e.g. Li, 2018), but that approach would leave only 454 usable rows
# here given the small number of Charged Off cases - not viable for this dataset.
default_statuses = ["Charged Off", "Late (31-120 days)", "Late (16-30 days)", "In Grace Period"]

df_lending["target"] = df_lending["loan_status"].isin(default_statuses).astype(int)
print(df_lending["target"].value_counts())
print(df_lending["target"].value_counts(normalize=True) * 100)

target
0    9822
1     178
Name: count, dtype: int64
target
0    98.22
1     1.78
Name: proportion, dtype: float64


In [24]:
# Drop loan_status (replaced by our binary target) and emp_title
# (4,742 unique free-text job titles - too high-cardinality to encode meaningfully)
df_lending = df_lending.drop(columns=["loan_status", "emp_title"])
print(df_lending.shape)

(10000, 55)


In [25]:
# Ordinal encoding for grade (A-G, natural order: A = best, G = worst)
grade_order = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4, "F": 5, "G": 6}
df_lending["grade"] = df_lending["grade"].map(grade_order)

# sub_grade follows the same pattern (A1-G5) - build the mapping programmatically
# rather than typing all 32 combinations by hand
sub_grade_order = {}
rank = 0
for letter in "ABCDEFG":
    for number in range(1, 6):
        sub_grade_order[f"{letter}{number}"] = rank
        rank += 1

df_lending["sub_grade"] = df_lending["sub_grade"].map(sub_grade_order)

print(df_lending[["grade", "sub_grade"]].isnull().sum())

grade        0
sub_grade    0
dtype: int64


In [26]:
# Frequency-encode state (50 categories - too many for clean one-hot encoding)
freq_map_state = df_lending["state"].value_counts(normalize=True)
df_lending["state"] = df_lending["state"].map(freq_map_state)

print(df_lending["state"].describe())

count    10000.000000
mean         0.049742
std          0.040771
min          0.001400
25%          0.018100
50%          0.033400
75%          0.079300
max          0.133000
Name: state, dtype: float64


In [27]:
# One-hot encode remaining low-cardinality nominal columns
nominal_cols_lending = [
    "homeownership", "verified_income", "verification_income_joint",
    "loan_purpose", "application_type", "issue_month",
    "initial_listing_status", "disbursement_method"
]
df_lending = pd.get_dummies(df_lending, columns=nominal_cols_lending, drop_first=True)
print(df_lending.shape)

(10000, 70)


In [28]:
# Check summary stats for continuous columns before deciding what to cap
continuous_cols_lending = [
    "annual_income", "debt_to_income", "loan_amount", "interest_rate",
    "installment", "balance", "paid_total", "paid_principal",
    "paid_interest", "total_credit_limit", "total_credit_utilized"
]
df_lending[continuous_cols_lending].describe()

,annual_income,debt_to_income,loan_amount,interest_rate,installment,balance,paid_total,paid_principal,paid_interest,total_credit_limit,total_credit_utilized
count,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,1.000000e+04,10000.000000
mean,7.922215e+04,19.304020,16361.922500,12.427524,476.205323,14458.916610,2494.234773,1894.448466,599.666781,1.836062e+05,51049.063100
std,6.473429e+04,14.987074,10301.956759,5.001105,294.851627,9964.561865,3958.230365,3884.407175,517.328062,1.876327e+05,53636.731172
min,0.000000e+00,0.000000,1000.000000,5.310000,30.750000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000
25%,4.500000e+04,11.067500,8000.000000,9.430000,256.040000,6679.065000,928.700000,587.100000,221.757500,5.159375e+04,19185.500000
50%,6.500000e+04,17.570000,14500.000000,11.980000,398.420000,12379.495000,1563.300000,984.990000,446.140000,1.146670e+05,36927.000000
75%,9.500000e+04,24.990000,24000.000000,15.050000,644.690000,20690.182500,2616.005000,1694.555000,825.420000,2.675500e+05,65421.000000
max,2.300000e+06,469.090000,40000.000000,30.940000,1566.590000,40000.000000,41630.443684,40000.000000,4216.440000,3.386034e+06,942456.000000


In [29]:
# Check how many rows have implausibly high debt_to_income values
print((df_lending["debt_to_income"] > 100).sum())
print(df_lending[df_lending["debt_to_income"] > 100]["debt_to_income"].describe())

33
count     33.000000
mean     177.074242
std       86.340066
min      100.720000
25%      128.610000
50%      144.740000
75%      191.700000
max      469.090000
Name: debt_to_income, dtype: float64


In [30]:
# Apply IQR-based outlier capping to continuous columns, floored at 0
# where negative values are nonsensical
continuous_cols_lending = [
    "annual_income", "debt_to_income", "loan_amount", "interest_rate",
    "installment", "balance", "paid_total", "paid_principal",
    "paid_interest", "total_credit_limit", "total_credit_utilized"
]

for col in continuous_cols_lending:
    Q1 = df_lending[col].quantile(0.25)
    Q3 = df_lending[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = max(0, Q1 - 1.5 * IQR)
    upper = Q3 + 1.5 * IQR
    n_capped = ((df_lending[col] < lower) | (df_lending[col] > upper)).sum()
    df_lending[col] = df_lending[col].clip(lower, upper)
    print(f"{col}: capped {n_capped} outlier values to range [{lower:.1f}, {upper:.1f}]")

annual_income: capped 508 outlier values to range [0.0, 170000.0]
debt_to_income: capped 221 outlier values to range [0.0, 45.9]
loan_amount: capped 0 outlier values to range [0.0, 48000.0]
interest_rate: capped 364 outlier values to range [1.0, 23.5]
installment: capped 208 outlier values to range [0.0, 1227.7]
balance: capped 0 outlier values to range [0.0, 41706.9]
paid_total: capped 619 outlier values to range [0.0, 5147.0]
paid_principal: capped 831 outlier values to range [0.0, 3355.7]
paid_interest: capped 401 outlier values to range [0.0, 1730.9]
total_credit_limit: capped 351 outlier values to range [0.0, 591484.4]
total_credit_utilized: capped 568 outlier values to range [0.0, 134774.2]


In [31]:
# Save the fully processed LendingClub dataset (target defined, encoded, outlier-capped)
df_lending.to_csv(DATA_DIR / "Lending Club Loan Dataset" / "loans_full_schema_encoded.csv", index=False)
print("Saved:", df_lending.shape)

Saved: (10000, 70)
